# Functional Programming II: Continuations

A continuation represents the computation that remains after a result becomes
available. By representing that remaining work as a function, we make control
flow explicit: it can be passed, composed, and invoked like other values.

The three lectures develop this idea in order:

- Part 1: what a continuation is, and how a closure represents it.
- Part 2: how direct-style control structures translate into CPS.
- Part 3: how explicit alternatives lead to a SAT and validity checker.

Run the notebook from the top. All executable examples use Python's standard
library; the circuit drawings are included with the release. Boolean N-queens
provides an optional extension without changing the three-part teaching path.


### Part 1: Make the remaining work explicit

In `inc(inc(x))`, after the inner call finishes, someone must increment its result.
We will represent that remaining work as a function.


#### Direct style vs CPS: first contact

In *direct style* we write functions that compute a result and return it:


```python
def inc(x):
    return x + 1

def twice(x):
    return inc(inc(x))
```


In *continuation-passing style* (CPS), a function passes its result onward. Instead
of returning that result to be used by its caller,
it receives an extra argument `k` (the *continuation*) that represents
"what to do with the result". It calls `k(result)` instead of `return result`.


In [ ]:
def inc_direct(x):
    return x + 1

def inc_cps(x, k):
    # k is the continuation: what to do with x+1
    k(x + 1)


def twice_direct(x):
    return inc_direct(inc_direct(x))

def twice_cps(x, k):
    # first increment x, then increment again, then pass to k
    inc_cps(x, lambda v1: inc_cps(v1, k))


def show(v):
    print("result:", v)

print("direct:", twice_direct(10))
twice_cps(10, show)


For `twice_cps(10, show)`, the first increment produces 11. Its continuation
receives 11 and starts the second increment; that produces 12, which is passed
to show. The intermediate continuation captures k, the final recipient.

These functions deliver the answer to k. Their Python return value is None;
the printed 12 is the computed answer, not the return value of twice_cps.

Key points:

- A CPS function takes an extra argument `k`.
- It passes its result to `k(value)` rather than leaving the caller to decide
  what to do next. Python still returns normally after these calls finish.
- To *run* a CPS function, you pass a *final continuation* (like `show`).


#### See the closure before hiding it in a lambda

The same twice_cps can be written with a named local function:


In [ ]:
def twice_named(x, k):
    def after_first(value):
        inc_cps(value, k)
    inc_cps(x, after_first)

twice_named(10, show)


after_first accepts the first result and retains k from the surrounding call.
This is last week's closure mechanism, now used to carry the remaining work.
The lambda in twice_cps is simply the shorter spelling of this local function.

The continuation is explicit, but Python still makes ordinary calls. Nothing
here yet saves stack space or explores more than one outcome.


#### Sequencing: name the work after each call

Consider a simple sequence of computations:


```python
def h(x):
    y = f(x)
    z = g(y)
    return z + 1
```


A CPS translation has to:

1. Call `f` in CPS form.
2. In its continuation, call `g`.
3. In its continuation, add 1 and call the final continuation.


In [ ]:
def f_direct(x):
    return x + 1

def g_direct(y):
    return y * 2

def h_direct(x):
    y = f_direct(x)
    z = g_direct(y)
    return z + 1

def f_cps(x, k):
    k(x + 1)

def g_cps(y, k):
    k(y * 2)

def h_cps(x, k):
    # CPS translation of "y = f(x); z = g(y); return z + 1"
    f_cps(x, lambda y:
          g_cps(y, lambda z:
                k(z + 1)))

print("h_direct:", h_direct(10))
h_cps(10, lambda v: print("h_cps:", v))


The CPS shape is always:

- replace `return` with `k(...)`
- replace `y ` f(...)= with `f_cps(..., lambda y: ...)`
- nest lambdas to represent "what to do next".

This is the *general scheme*: for each step, we introduce a continuation
describing the remaining steps.


#### Trace the captured work

In `h_cps` above, the continuation `lambda y: g_cps(y, ...)` describes the work
after f finishes. The innermost continuation captures `k` so that, after adding
one, it can pass the answer to the original recipient.

Before running the following, predict the two results. The computation is the
same; only the recipient of its result changes.


In [ ]:
h_cps(10, lambda value: print(value))
h_cps(10, lambda value: print(value == 23))


The first prints 23; the second prints True. The continuation is not necessarily
a printer: it may start another computation or test a result.

These nested continuations are the remaining work at successive call sites.
Part 2 applies this same construction systematically to other control structures.


#### A complete expression: the hypotenuse

For the hypotenuse of a right triangle, compute the two squares, add them,
then take the square root. Each intermediate result becomes the argument of
the next continuation.


In [ ]:
def square_cps(x, k):
    return k(x * x)

def sum_cps(x, y, k):
    return k(x + y)

def root_cps(x, k):
    return k(x ** 0.5)

def hypotenuse_cps(x, y, k):
    return square_cps(x, lambda x2:
        square_cps(y, lambda y2:
            sum_cps(x2, y2, lambda total:
                root_cps(total, k))))

hypotenuse_cps(6, 8, show)


Trace the values: 36, then 64, then 100, then 10. The closure waiting for y2
retains x2, so the two squares can be added. This is why closures are useful
here: each stage carries exactly the values needed by the work that follows.


### Part 2: Translate control structures into CPS

CPS is not restricted to arithmetic examples. We can translate a program by
replacing the caller's implicit remaining work with an explicit continuation.
The rules below cover values, calls, sequencing, conditionals, loops, recursion,
and a success/error model of exceptions. They are not a complete Python
interpreter: features such as finally and generator suspension need additional
rules.


#### A general scheme for translating to CPS

We sketch simple *translation rules* that you can apply mechanically.

We restrict to expressions and functions; this already covers a large
class of programs.


##### constants and variables

Direct style:


```python
42
x
```


CPS form, given a continuation `k`:


```python
k(42)
k(x)
```


We always *call* the continuation with the value.


##### binary operations

Direct style:


```python
x + y
```


CPS form, assuming `x` and `y` are already values:


```python
k(x + y)
```


If `x` and `y` themselves come from computations, we CPS-transform those
computations and chain them using nested continuations.


##### function definition and call

Direct style:


```python
def f(x):
    return e

result = f(10)
```


CPS form:

1. Translate the body of `f`:

   - direct: `return e`
   - CPS:    `CPS(e, k)` i.e. "evaluate e then call k on the result"

2. The CPS version of `f` takes an extra continuation argument:


```python
def example_f_cps(x, k):
    # CPS(e, k)
    ...
```


3. A call `f(10)` in direct style becomes `example_f_cps(10, k)` in CPS.


In [ ]:
def example_f_direct(x):
    return (x + 1) * 2

def example_f_cps(x, k):
    # mechanically: compute (x+1)*2 then call k
    k((x + 1) * 2)


def example_direct():
    return example_f_direct(10)

def example_cps(k):
    # translate "return example_f_direct(10)" as:
    example_f_cps(10, k)

print("example_direct:", example_direct())
example_cps(lambda v: print("example_cps:", v))


Note: We did *not* change the arithmetic; we only replaced `return` with
"call the continuation". In more complex programs, we systematically introduce
continuations at each call site.


#### Conditionals: choose a continuation

An if–then–else selects one of two paths. Represent each path by a continuation
that includes everything to do along that path:


In [ ]:
def choose_cps(condition, then_k, else_k):
    if condition:
        return then_k()
    return else_k()


The condition is already a Boolean value. Only the selected continuation runs.
These continuations take no arguments: their closures capture the values they need.


In [ ]:
def magnitude_direct(x):
    if x >= 0:
        return x
    else:
        return -x

def magnitude_cps(x, k):
    return choose_cps(x >= 0,
                      lambda: k(x),
                      lambda: k(-x))

magnitude_cps(-3, show)


Both branch closures capture k, the common work after finding the magnitude.
There is no need to pass it separately to choose_cps. For example:


In [ ]:
magnitude_cps(-3, lambda value: value + 10)


The result is 13. The negative branch delivers 3 to the continuation, which
adds 10. The positive branch is never called.

Here `return k(...)` forwards the recipient's Python return value. The computed
answer still reaches its destination through k. In Part 1 we ignored that
return value because show only printed; both forms deliver the same answer.


#### Loops: express repetition with recursion

A loop can be represented by recursion, carrying its changing state in function
arguments. We can then apply the same CPS transformation used for other calls.
Converting a loop to recursion is not, by itself, a CPS transformation.

Start with a loop that sums a sequence of integers:


In [ ]:
def total_loop(xs):
    result = 0
    for value in xs:
        result += value
    return result


An equivalent recursive sum adds the current element to the sum of the remaining
elements. Here i is the index of the next element to include:


In [ ]:
def total(xs, i):
    if i == len(xs):
        return 0
    return xs[i] + total(xs, i + 1)

total_loop([3, -2, 5]), total([3, -2, 5], 0)


Both results are 6. After the recursive call returns, we still need to add xs[i].
Make that remaining work a continuation:


In [ ]:
def total_cps(xs, i, k):
    if i == len(xs):
        return k(0)
    return total_cps(
        xs, i + 1,
        lambda subtotal: k(xs[i] + subtotal)
    )

total_cps([3, -2, 5], 0, lambda result: result)


Each lambda captures its own i and k, along with xs. When the base case delivers
0, the saved continuations add 5, then -2, then 3, producing 6. Finally, the
original k receives the answer. For example, replacing it with
`lambda result: result * 10` produces 60 without changing the summation.

Recursion expresses the repetition; CPS makes the remaining computation explicit.
These versions still use Python's call stack. Later we will use a trampoline
to avoid the recursion-depth limit.


#### Recursion: retain the work after the recursive call

For factorial, the remaining work after factorial(n-1) is to multiply by n.
The transformation is the same as for the recursive sum: capture that pending
operation in a continuation.


In [ ]:
def factorial_direct(n):
    if n == 0:
        return 1
    return n * factorial_direct(n - 1)

def factorial_direct_cps(n, k):
    if n == 0:
        return k(1)
    return factorial_direct_cps(n - 1, lambda value: k(n * value))

factorial_direct(3), factorial_direct_cps(3, lambda result: result)


Each new continuation captures its own n and the previous k. At the base case,
the result passes through the saved multiplications: `1 -> 1 -> 2 -> 6`.
The control that the ordinary call stack remembered is now also explicit in
the closure chain.


#### Exceptions: choose where failure goes

An ordinary call has a normal outcome and may also fail. Give the CPS
computation two continuations: on_return for a result and on_error for an error.
A handler changes the error continuation for the operation it protects.

Here a computation has the interface `operation(on_return, on_error)`.
We define four operations that

- deliver a value

- deliver an error

- sequence two computations

- try a computation and, on failure, run a recovery computation


In [ ]:
def return_(x):
  return lambda on_return, on_error: on_return(x)

def raise_(x):
  return lambda on_return, on_error: on_error(x)

def bind(op1, op2):
  """ executes op1 then op2, outputs error if either fails """
  return lambda on_return, on_error: op1(lambda res: op2(res)(on_return, on_error), on_error)

def try_(op1, op2):
  """ executes op1, and if op1 fails, executes op2 """
  return lambda on_return, on_error: op1(on_return, lambda res: op2(res)(on_return, on_error))


In bind, op1 is a computation; op2 is a function that uses its successful result
to construct the next computation. The same error continuation is passed through,
so failure bypasses the remaining normal work.

In try_, the replacement error continuation constructs the recovery computation.
That recovery uses the outer error continuation: a failure inside the handler
is not caught again by the same handler.

For demonstration, our final recipients print the result or error:


In [ ]:
on_return = lambda x: print('ok: ', x)
on_error = lambda x: print('error: ', x)

run_cps = lambda op: op(on_return, on_error)


and a function that might produce an error, depending on its parameters,


In [ ]:
def safe_div(x,y):
  def f(on_return, on_error):
    if y==0:
      return on_error('div by zero')
    else:
      return on_return(x/y)
  return f


In the next example, we apply two safe division commands in sequence, first divide $x/2$ then divide the result by $y$,


In [ ]:
op = lambda x,y: bind(safe_div(x,2),
                      lambda a: safe_div(a,y))
run_cps(op(30,2))
run_cps(op(30,0))


A handler can recover: try to divide $x/y$, and on failure divide $x/10$.
The direct-style idea is:


```python
try:
    answer = x / y
except ZeroDivisionError:
    answer = x / 10
```


In the CPS version, try_ supplies the replacement error continuation:


In [ ]:
op = lambda x,y: try_(safe_div(x,y),
                      lambda _: safe_div(x,10))

run_cps(op(30,0))


The next example propagates the error to the outer recipient. It models re-raising;
it does not execute Python's raise statement:


In [ ]:
op = lambda x,y: try_(safe_div(x,y),
                      lambda msg: raise_(msg))

run_cps(op(30,2))
run_cps(op(30,0))


#### Executing CPS in Python: thunks and trampolines

A continuation receives a result. A *thunk* takes no arguments and represents
a step that has not run yet. Instead of immediately calling the next step,
we can return a thunk and let a loop call it.

Like the curried functions from last week, a thunk can be a closure that captures
values for later use. The purposes differ: currying supplies arguments in stages;
a thunk postpones a computation. A trampoline executes the postponed steps.


In [ ]:
end_cont = lambda value: value


Let's implement a function $f(x,y)$ that computes $2x+y$ in CPS-style.

For that, we create a sum and a multiplication function, that are called in sequence by `f`,


In [ ]:
def add(x, y, cont=end_cont):
  return lambda: cont(x+y)

def mul(x, y, cont=end_cont):
  return lambda: cont(x*y)

# compute 2*x + y
def f(x, y, cont=end_cont):
  cont2 = lambda new_val: add(new_val, y, cont)
  return mul(2, x, cont2)


If we try to evaluate `f` for some input,


In [ ]:
f(3, 10)


it returns a thunk: a function of no arguments that represents the next step.
Calling that thunk performs the multiplication and passes 6 to cont2. It returns
the addition thunk. Calling that second thunk adds 10 and passes 16 to end_cont:


In [ ]:
f(3, 10)()()


A trampoline repeatedly calls each returned thunk until it gets a final value:


In [ ]:
def trampoline(f, *args):
  """ iteratively invokes thunk-returning functions """
  v = f(*args)
  while callable(v):
    v = v()
  return v


Returning a thunk lets the current Python call finish before the next step
starts. In this small implementation, a callable always means another step;
the final answer must therefore be a non-callable value.

To compute the final result using a trampoline:


In [ ]:
trampoline(f, 3, 10)


##### Tail calls

In a tail call, the caller has no work left except to return the called function's
result. A language implementation with tail-call optimization (TCO) can reuse
the caller's stack frame rather than keep it waiting. Python does not perform
this optimization.

This helper prints stars to show how the call stack grows during recursion.
We record a baseline once, before running the examples.


In [ ]:
import traceback

INITIAL_DEPTH = len(traceback.extract_stack())

def show_depth(*state):
    depth = len(traceback.extract_stack()) - INITIAL_DEPTH
    print("*" * depth, ":", state)


Let's test it using a standard tail-recursive implementation of the factorial function,


In [ ]:
def fact_tail(n, fac=1):
  show_depth(n)
  if n==0:
    return fac
  return fact_tail(n-1, n*fac)

print(fact_tail(8))


The stars show the stack growing: each caller waits for the recursive call to
return, even though it has no further calculation to do.

Now change only the last line: return a thunk instead of making the recursive
call immediately.


In [ ]:
def fact_step(n, fac=1):
  show_depth(n)
  if n == 0:
    return fac
  return lambda: fact_step(n-1, n*fac)

print(trampoline(fact_step, 8))


The current call finishes before the trampoline invokes the returned thunk.
For `trampoline(fact_step, 3)`, the successive `(n, fac)` states are
`(3, 1) -> (2, 3) -> (1, 6) -> (0, 6)`, followed by the answer 6.
Each step returns to the trampoline instead of leaving a chain of callers
waiting on the stack.

This gives bounded call-stack depth, not necessarily faster execution. Actual
TCO can replace a tail call with a jump. Our Python trampoline still creates
closures and makes function calls; it can add overhead. It avoids stack growth
without changing Python's treatment of tail calls.


##### When there is work after the recursive call

The accumulator above carries a running product, so no multiplication remains
after the recursive call. The earlier factorial definition instead computed
`n * factorial(n-1)`. Its CPS translation stores those pending multiplications
in continuations. We can run that version with a trampoline too.

For factorial, the work after computing factorial(n-1) is to multiply its answer
by n and deliver that product to cont. Thus `lambda fac: cont(n*fac)` describes
the pending work. To defer that work too, it becomes
`lambda fac: lambda: cont(n*fac)`.

For n=3, descent saves the multiplications by 3, 2 and 1. The base case supplies
1, and those saved continuations produce 1, then 2, then 6. Both descent and
the saved work must take steps through the trampoline:


In [ ]:
def fact_cps(n, cont=end_cont):
  show_depth(n)
  if n == 0:
    return lambda: cont(1)
  return lambda: fact_cps(n-1, lambda fac: lambda: cont(n*fac))

print(trampoline(fact_cps, 8))


Both the recursive descent and the continuation chain return thunks (functions
of no arguments). The trampoline calls each thunk after the previous call has
returned, so neither phase builds a growing Python call stack. Pending work
still occupies memory in closures; we have not made Python optimize tail calls.


##### Check the distinction

- `cont(value)` performs the continuation now.
- `lambda: cont(value)` packages that call for later.
- The trampoline runs one packaged step at a time.

In `fact_cps`, the closure chain remembers the pending multiplications.
In `fact_step`, the accumulator already contains the product processed so far. Both avoid a
growing Python call stack under the trampoline, but they store pending work
differently.

Why would returning a thunk for recursive descent alone be insufficient?
The chain of pending continuations must also defer its calls; otherwise it
can grow the stack when the base case is reached.


### Part 3: Explore the alternatives in one execution

A normal function returns one answer. A search may find none, one, or many.
We can express this by calling its continuation for each successful branch.
A failed branch produces no result.

Our goal is to check a proposed circuit rewrite: does it preserve the output
for every input, or can we find an input that exposes a bug? We start with
member, then make the instruction "try another possibility" explicit.


#### First, deliver every candidate

The idea is:

- if a search does not produce a result, do not call the continuation

- if a search produces several results, call the continuation for each result

- pass the remaining work into each branch

This first example executes a continuation for each element of the given list:


In [ ]:
def member(xs, cont):
  if xs:
    x, *xs = xs
    cont(x)
    return member(xs, cont)


Let's choose a print as the continuation:


In [ ]:
cont_show = lambda *x: print(*x, end='; ')
member([1,2,3], cont_show)


Searches can be combined:


In [ ]:
def search2(xs, ys):
  member(xs,
         lambda x: member(ys,
                          lambda y: cont_show(x,y)))

search2([1,2,3], 'abcd')


And we can apply filters that cut the search if some predicate is not satisfied:


In [ ]:
cont_nothing = lambda x: None # do nothing continuation

def filter_callback(p, cont):
  return lambda *x: cont(*x) if p(x[0]) else cont_nothing(x)

def search3(xs, ys):
  member(xs,
         lambda x: member(ys,
                          lambda y: filter_callback(lambda x: x%2==0, cont_show)(x,y)))

search3([1,2,3], 'abcd')


The call to member does not stop after the first candidate. After its
continuation finishes, it visits the next candidate. In search2, each x
starts a search through all y values. One call explores every pair.
Filtering simply declines to continue a rejected branch.

An ordinary if chooses one branch for its current input. We have not changed
that meaning: we explicitly visit alternatives, and each visit can make its
own decisions.


#### Let the continuation request another candidate

Our first member visits every candidate automatically. For the logic checker,
we want the rest of the computation to decide: accept this candidate, or ask
for another. Pass it a second function, backtrack:


In [ ]:
def member_backtrack(xs, yes, no):
    if not xs:
        return no()
    x, *rest = xs

    def backtrack():
        return member_backtrack(rest, yes, no)

    return yes(x, backtrack)


Here `yes(value, backtrack)` receives a candidate and the remaining search.
Calling backtrack resumes the remaining alternatives; it does not repeat the
same attempt. Calling no means this search has
exhausted its candidates. Both backtrack and no take no arguments because their
closures already retain what they need.


In [ ]:
def accept_even(value, backtrack):
    if value % 2 == 0:
        return value
    return backtrack()

def exhausted():
    return "no answer"

member_backtrack([1, 3, 4, 6], accept_even, exhausted)


The result is 4. The continuation rejects 1 and 3 by calling backtrack, then
accepts 4 without requesting 6. To visit every candidate instead, a success
continuation can record the value and then call backtrack.

This resembles our exception example, but failure now means "resume an
alternative" rather than "report an error". A later computation can reject a
candidate, so success must carry the backtrack continuation along with the value.


#### From candidates to Boolean assignments

An engineer replaces a combinational circuit with a lower-transistor-count
implementation. Testing a few inputs cannot
establish that the new circuit agrees with the old one on every input. Instead,
ask a different question: *is there any input on which their outputs disagree?*

Consider a multiplexer: when the selector S is true, its output is A;
otherwise its output is B. Its Boolean expression is

`(S and A) or (not S and B)`.

A NAND gate computes `not (x and y)`. By De Morgan's law, the same multiplexer
can be implemented as `NAND(NAND(S, A), NAND(NOT(S), B))`:
negating the AND of the two complemented product terms gives their OR.

Why make this change? Use a simple static-CMOS cost model: a two-input NAND
uses 4 transistors; a two-input AND or OR uses 6; an inverter uses 2.
The AND and OR implementations include an output inverter after NAND or NOR.
These are gate-by-gate counts, not a claim about the best possible multiplexer.

| Implementation | Gates | Transistors |
|----------------|-------|-------------|
| Original | 2 AND ~~ 1 OR ~~ 1 NOT | 6 ~~ 6 ~~ 6 + 2 = 20 |
| NAND implementation | 3 NAND ~~ 1 NOT | 4 ~~ 4 ~~ 4 ~~ 2 = 14 |

The proposed change saves 6 transistors (30%) under this model. It does not by
itself establish a speed or power improvement: loading, transistor sizing and
the available cell library also matter. For the underlying gate construction,
see [MIT's CMOS gates lecture](https://csg.csail.mit.edu/6.375/6_375_2006_www/handouts/lectures/L04-CMOS.pdf).

The two networks below use the same input signals. A small output circle means
inversion: an AND-shaped gate with this circle is NAND.

![Original multiplexer: two AND gates feed OR, costing 20 transistors including the inverter. Optimized implementation: three NAND gates and an inverter cost 14 under the stated static-CMOS model.](images/sat-mux-circuits.png)

To check the rewrite, construct
the exclusive-or of the two output expressions. XOR is true exactly when
its inputs differ. An assignment making this disagreement formula true is
a concrete bug-revealing test case. If none exists, the two circuits agree
for every Boolean input.

Finding an assignment that makes a Boolean formula true is /Boolean
satisfiability/ (SAT). We describe the required property; the solver searches
for a witness. A *valid* formula, or tautology, is true for every assignment.
Equivalence can be checked either by proving agreement valid or by showing
that disagreement is unsatisfiable. We will return to the circuit once the
checker is built.

This adapts [Denys Duchier’s checker](https://www.ps.uni-saarland.de/~duchier/python/continuations.html). Its interface follows the
same pattern as member_backtrack:

- `formula.satisfy(environment, yes, no)` searches for a way to make it true.
- `formula.falsify(environment, yes, no)` searches for a way to make it false.
- environment is a dictionary of truth values chosen so far, such as
  `{"P": False}`. Unmentioned variables are still unconstrained.
- `yes(environment, backtrack)` receives a consistent partial assignment and a
  function for resuming the remaining search.
- `no()` is called when no possibility remains in this search.

A partial assignment is enough when the formula is already guaranteed true
(or false) regardless of the unassigned variables. We need not enumerate every
full truth-table row.

The control rules follow directly from Boolean logic:

| Formula | To make it true | To make it false |
|---------|-----------------|------------------|
| not P   | Falsify P       | Satisfy P        |
| P and Q | Satisfy both    | Falsify either   |
| P or Q  | Satisfy either  | Falsify both     |

"Both" connects computations with a success continuation. "Either" remembers
the second alternative in a failure continuation. The classes below implement
these two patterns; constructors and string methods only represent formulas.

The base class provides isValid: finding a counterexample returns False;
exhausting the search returns True. We first use direct function calls so the
success and failure paths are easy to follow. The optional N-queens example
will expose the stack limitation and motivate returning to the trampoline.


In [ ]:
# Adapted from Denys Duchier's continuation-based logic checker.
# Copyright (c) Feb 2000, by Denys Duchier, Universitaet des Saarlandes
class Formula:
  def isValid(self):
    """a formula is valid iff it cannot be falsified"""
    def counterexample(environment, backtrack):
      return False

    def no_counterexample():
      return True

    return trampoline(self.falsify, {}, counterexample, no_counterexample)

  _tracing = None  # Set to print to display search steps.

  # Public entry points delegate to the search rules.
  def satisfy(self, environment, yes, no):
    if Formula._tracing is not None:
      Formula._tracing(f'satisfy {self} environment:{environment}')
    return self._satisfy(environment, yes, no)

  def falsify(self, environment, yes, no):
    if Formula._tracing is not None:
      Formula._tracing(f'falsify {self} environment:{environment}')
    return self._falsify(environment, yes, no)


##### Variables: extend an assignment or reject a conflict

A variable can be assigned a truth value unless that contradicts an earlier
choice. We copy the dictionary before extending it, so trying one branch does
not change the starting assignment of another. The method bind here simply
extends a dictionary; it is not the sequencing combinator from Part 2.


In [ ]:
class Variable(Formula):
  def __init__(self,v):
    self.v = v

  def __str__(self):
    return self.v

  def bind(self, value, environment):
    """returns a new partial assignment that additionally
       assigns the truth 'value' to this propositional variable"""
    environment = environment.copy()
    environment[self.v] = value
    return environment

  def assign(self, value, environment, yes, no):
    """Continue with a consistent assignment, or backtrack on a conflict."""
    if self.v in environment:
      if environment[self.v] != value:
        return no()
      return yes(environment, no)
    return yes(self.bind(value, environment), no)

  def _satisfy(self, environment, yes, no):
    """to satisfy a propositional variable, we must assign it true"""
    return self.assign(True, environment, yes, no)

  def _falsify(self, environment, yes, no):
    """to falsify a propositional variable, we must assign it false"""
    return self.assign(False, environment, yes, no)


##### Compound formulas: sequence or try an alternative

Negation just swaps satisfy and falsify. In Conjunction, read after_left as
"having satisfied the left side, try the right side with that assignment".
If the right side fails, backtrack_left resumes alternatives from the left side.
In Disjunction, try_right means "the left alternative failed; try the right
from the original environment".


In [ ]:
class Negation(Formula):
  def __init__(self, p):
    self.p = p

  def __str__(self):
    return f'¬{self.p}'

  def _satisfy(self, environment, yes, no):
    """to satisfy ¬P we must falsify P"""
    return self.p.falsify(environment, yes, no)

  def _falsify(self, environment, yes, no):
    """to falsify ¬P we must satisfy P"""
    return self.p.satisfy(environment, yes, no)


In [ ]:
class Conjunction(Formula):
  def __init__(self, p, q):
    self.p = p
    self.q = q

  def __str__(self):
    return f'({self.p} ∧ {self.q})'

  def _satisfy(self, environment, yes, no):
    """to satisfy P∧Q we must satisfy both P and Q"""
    def after_left(assignment, backtrack_left):
      return self.q.satisfy(assignment, yes, backtrack_left)

    return self.p.satisfy(environment, after_left, no)

  def _falsify(self, environment, yes, no):
    """to falsify P∧Q we can falsify either P or Q"""
    def try_right():
      return self.q.falsify(environment, yes, no)

    return self.p.falsify(environment, yes, try_right)


In [ ]:
class Disjunction(Formula):
  def __init__(self,p,q):
    self.p = p
    self.q = q

  def __str__(self):
    return f'({self.p} ∨ {self.q})'

  def _satisfy(self, environment, yes, no):
    """to satisfy P∨Q we can satisfy either P or Q"""
    def try_right():
      return self.q.satisfy(environment, yes, no)

    return self.p.satisfy(environment, yes, try_right)

  def _falsify(self, environment, yes, no):
    """to falsify P∨Q we must falsify both P and Q"""
    def after_left(assignment, backtrack_left):
      return self.q.falsify(assignment, yes, backtrack_left)

    return self.p.falsify(environment, after_left, no)


The next auxiliary functions are useful to facilitate the writing of logical formulas:


In [ ]:
def AND(*args):
  """n-ary version of Conjunction"""
  formula, *args = args
  for x in args:
    formula = Conjunction(formula,x)
  return formula

def OR(*args):
  """n-ary version of Disjunction"""
  formula, *args = args
  for x in args:
    formula = Disjunction(formula,x)
  return formula

def NOT(x):
  return Negation(x)

def IF(p,q):
  return OR(NOT(p),q)

def IFF(p,q):
  return AND(IF(p,q), IF(q,p))

def XOR(p,q):
  return OR(AND(p,NOT(q)), AND(NOT(p),q))


##### Watch a failed choice lead to a solution

Before checking the circuit, trace the smaller formula `(P or Q) and not P`.
It is small enough to see exactly where a candidate is rejected and retried.


In [ ]:
P = Variable('P')
Q = Variable('Q')
R = Variable('R')

def first_assignment(environment, backtrack):
    return environment

def no_assignment():
    return None

def solve_sat(formula):
    return trampoline(formula.satisfy, {}, first_assignment, no_assignment)

def find_counterexample(formula):
    return trampoline(formula.falsify, {}, first_assignment, no_assignment)

requirement = AND(OR(P, Q), NOT(P))
solve_sat(requirement)


solve_sat asks for the first satisfying assignment; find_counterexample asks
for the first falsifying assignment. Both use our trampoline as the outer runner.
For now the search makes its calls directly, so the trampoline just receives
its final answer. Later it will also execute deferred steps. This alone does
not prevent stack growth inside the direct-recursive search.

The answer is `{"Q": True, "P": False}`. Follow the search:

1. OR first satisfies P, producing `{"P": True}`. It remembers trying Q
   as the alternative.
2. AND continues with NOT(P). This requires P=False and conflicts with P=True.
3. The conflict invokes the saved alternative: try Q from the original empty
   environment, not from the rejected P=True assignment.
4. Q=True followed by P=False succeeds. first_assignment accepts the result
   without asking for another answer.

The important step is 3: a failure *later* in the computation resumes a choice
made *earlier*. The backtrack closure explicitly remembers that choice point.


In [ ]:
solve_sat(AND(P, NOT(P))) is None


This yields True: no consistent assignment satisfies both P and NOT(P).


##### From finding an assignment to checking a theorem

Suppose at least one of P and Q holds, and each implies R. Must R hold?
Here IF(P, R) represents logical implication, equivalent to `not P or R`;
it is not Python's if statement.


In [ ]:
form = IF(AND(OR(P, Q), IF(P, R), IF(Q, R)), R)
form.isValid()


The answer is True. isValid searches for a counterexample by calling falsify.
Finding even one would return False. Exhausting every alternative returns True.
Compare this with a claim that does have a counterexample:


In [ ]:
find_counterexample(IF(P, Q))


The answer is `{"P": True, "Q": False}`. Thus IF(P, Q) is not valid, even
though it is satisfiable. A satisfying assignment establishes possibility;
absence of any counterexample establishes validity.


##### Application: verify an optimization and generate a failing test

Return to our multiplexer. We encode the original and proposed replacement
using the same input variables:


In [ ]:
S = Variable('S')
A = Variable('A')
B = Variable('B')

original = OR(AND(S, A), AND(NOT(S), B))

def NAND(x, y):
    return NOT(AND(x, y))

replacement = NAND(NAND(S, A), NAND(NOT(S), B))
disagreement = XOR(original, replacement)

solve_sat(disagreement) is None


The result is True: no input makes their outputs differ. This establishes
equivalence for this Boolean model, not just agreement on selected tests.
The solver establishes preserved behavior; the separate cost model establishes
the transistor saving. NAND expands into NOT and AND here only to describe its
Boolean meaning to the checker, not to prescribe its physical implementation.

Now suppose the first NAND is accidentally wired to B instead of A. The gate
count is still smaller, but is the circuit still correct? Ask the same question:


In [ ]:
faulty = NAND(NAND(S, B), NAND(NOT(S), B))
counterexample = solve_sat(XOR(original, faulty))
counterexample


The checker finds `{"S": True, "A": True, "B": False}`. Connect the
same inputs to both circuits and XOR their outputs. This comparison circuit is
often called a *miter*: output 1 means the circuits disagree.

![The same inputs S=1, A=1, B=0 produce output 1 in the original and 0 in the faulty replacement. XOR outputs 1, exposing the mismatch.](images/sat-counterexample.png)

Apply that input
to ordinary Python versions of the circuits:


In [ ]:
def mux_original(s, a, b):
    return (s and a) or (not s and b)

def nand_value(x, y):
    return not (x and y)

def mux_optimized(s, a, b):
    return nand_value(nand_value(s, a), nand_value(not s, b))

def mux_faulty(s, a, b):
    return nand_value(nand_value(s, b), nand_value(not s, b))

s = counterexample['S']
a = counterexample['A']
b = counterexample['B']
mux_original(s, a, b), mux_optimized(s, a, b), mux_faulty(s, a, b)


The outputs are `(True, True, False)`: the original and correct optimization
agree, but the faulty version differs. A test case generated from the specification
has exposed the bug. In general, the checker may return a partial assignment;
unmentioned inputs can then take either value. This example fixes all three.

The same idea supports three tasks: constraints describe acceptable decisions;
a disagreement formula searches for a verification counterexample; a witness
becomes a test input. In the optional N-queens example, we will describe legal
boards with Boolean constraints and reuse this engine without adding a
queen-specific search algorithm.

We have expressed a small propositional theorem checker using ordinary Python
functions and closures. It searches sequentially, not in parallel. Direct
calls still grow Python's stack, and large searches can be expensive;
this is a demonstration of control, not an industrial SAT solver.


### Recap

The central idea is that *control can be explicit*.

1. A continuation represents the work after a result becomes available.
   A closure retains the values that this work needs.
2. Direct-style calls, sequencing, branches, iteration, recursion, and exception
   handling can be expressed by passing the appropriate continuations.
3. Thunks and a trampoline separate describing a next step from executing it.
4. A search can continue after a branch fails or succeeds. Its continuations
   let us stop at a satisfactory answer or exhaust a finite search to establish
   that no answer exists; conflicting branches can be pruned.

The striking expression style is visible in the equivalence check:


```python
solve_sat(XOR(original, replacement))
```


We describe disagreement; the composed success and failure continuations
control the search for a counterexample. A success can carry the remaining
alternatives forward, and a later failure can return to them. Search stops when
a counterexample is found, or exhausts the alternatives to establish equivalence.
The compact expression does not make that search work disappear.

Last week, closures let functions retain data. Here they retain what to do next.
Control is no longer only implicit in the placement of statements: it is
something our program can represent and compose.


### Optional extensions

Boolean N-queens is the sole extension: reuse the checker for a new problem,
observe its execution limit, and repair it without changing the logical rules.


#### N-queens as Boolean constraints

Place n queens on an n-by-n board so that no two share a row, column, or
diagonal. Instead of writing a new search procedure, introduce a Boolean
variable for each square: `q[row][column]` is true when that square has a queen.
Rows and columns are numbered from 0 to n-1.

We need only two kinds of requirement:

1. Each column contains at least one queen: OR together its n variables.
2. Two attacking squares cannot both contain queens: for their variables
   x and y, require `not x or not y`.

Attacking pairs include squares in the same column, so the two requirements
together give exactly one queen per column, hence n queens in total. Row and
diagonal conflicts are excluded by the same pairwise rule.


In [ ]:
def not_both(x, y):
    return OR(NOT(x), NOT(y))

def queen_constraints(n):
    # n must be a positive integer.
    q = [[Variable(f'q_{r}_{c}') for c in range(n)] for r in range(n)]
    constraints = []

    for c in range(n):
        constraints.append(OR(*[q[r][c] for r in range(n)]))

    squares = [(r, c) for r in range(n) for c in range(n)]
    for i, (r, c) in enumerate(squares):
        for rr, cc in squares[i + 1:]:
            if r == rr or c == cc or abs(r - rr) == abs(c - cc):
                constraints.append(not_both(q[r][c], q[rr][cc]))

    return AND(*constraints)

def queen_board(n, assignment):
    return [[int(assignment.get(f'q_{r}_{c}', False))
             for c in range(n)] for r in range(n)]


##### First attempt: the encoding is correct, but the stack grows

The loops above build the formula; they do not search board configurations.
Try the direct-recursive checker on a four-by-four board:


In [ ]:
recursive_board_formula = queen_constraints(4)
stack_overflow_seen = False
try:
    solve_sat(recursive_board_formula)
except RecursionError:
    stack_overflow_seen = True
    print("RecursionError: the direct search exhausted Python's call stack.")


At Python's usual recursion limit, this reports RecursionError. The catch is
only here to let Run All continue to the repair; it is not part of the solver.
The precise failure point depends on the Python environment. This is not proof
that the formula is unsatisfiable: the implementation ran out of stack before
answering the question.

Why does a small board cause trouble? Following clauses and backtracking to earlier
choices makes nested calls. Making control explicit did not make Python
optimize those calls. We have encountered exactly the problem from Part 2.


##### Fix 1: return search steps to the trampoline

All recursive transitions already pass through satisfy or falsify. We only
need these two entry points to return a thunk instead of performing the call
immediately. The logical rules in Variable, Negation, Conjunction and
Disjunction stay exactly as they are.

Recall decorators: a wrapper can change how a function is called without
changing its body. This wrapper postpones a call until its returned thunk is
invoked:


In [ ]:
def defer_call(method):
    def deferred(self, environment, yes, no):
        return lambda: method(self, environment, yes, no)
    return deferred

Formula.satisfy = defer_call(Formula.satisfy)
Formula.falsify = defer_call(Formula.falsify)


These two assignments apply the wrapper to the existing methods, just as
`@defer_call` would when defining them. All formula objects inherit these
methods, including objects we already constructed. There are no new formula
classes and no changes to the Boolean rules.

Now solve_sat's trampoline receives a thunk, calls it, and repeats. When a
search step invokes another formula's satisfy or falsify, that call returns
the next thunk. The current call can finish before the next step begins.
The existing circuit examples and isValid use the same outer runner and
continue to work after this change.


##### Fix 2: avoid exploring the same case twice

A trampoline removes stack growth, not search work. In `not x or not y`,
the case x=False, y=False is reached through both alternatives. Across many
clauses, these overlapping paths cause substantial repeated search. We do not
run that expensive intermediate version here.

Instead list the three allowed truth-table rows explicitly: `(False, False)`,
`(True, False)`, and `(False, True)`. These cases do not overlap. This changes
how the formula is searched, but not which assignments satisfy it.


In [ ]:
def not_both(x, y):
    return OR(AND(NOT(x), NOT(y)),
              AND(x, NOT(y)),
              AND(NOT(x), y))

board_assignment = solve_sat(queen_constraints(4))
queen_board(4, board_assignment)


Now we obtain a board: a 1 marks a queen and a 0 an empty square. The decoder
sets any unconstrained variable to False; every extension of a returned partial
assignment satisfies the formula. The two fixes addressed different problems:
the trampoline bounds the call stack; disjoint cases reduce repeated work.

For a three-by-three board there is no solution:


In [ ]:
solve_sat(queen_constraints(3)) is None


This returns True. The same engine has now checked circuit equivalence,
generated a failing test, and solved a placement problem. What changed was
the formula, not the search mechanism. Keep these boards small: this teaching
engine does not include modern SAT optimizations, and its search can grow rapidly.


## References

The CPS translation material and examples are adapted from the earlier course
notes. The logic checker adapts Denys Duchier's continuation-based search.

- [Denys Duchier: Continuations in Python](https://www.ps.uni-saarland.de/~duchier/python/continuations.html)
- [Eli Bendersky: On Recursion, Continuations and Trampolines](https://eli.thegreenplace.net/2017/on-recursion-continuations-and-trampolines/)
